In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码。
设置 os.environ['HF_ENDPOINT'] = \'https://hf-mirror.com' ，这样做具体改变了什么？
为什么要设置HF_ENDPOINT=\'https://hf-mirror.com'而非直接使用官方源？
dataset = datasets.load_dataset("bentrevett/multi30k") 这行代码具体完成了什么操作？

### 代码1：`os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
#### 1. 具体改变
这行代码通过修改系统环境变量，**将Hugging Face生态库（`datasets`、`transformers`等）的默认数据/模型下载源，从海外官方源（`https://huggingface.co`）全局切换为国内镜像源 `hf-mirror.com`**，所有后续Hugging Face相关请求都会走这个镜像地址。

#### 2. 为什么用镜像源而非官方源
- **解决网络限制**：Hugging Face官方服务器位于海外，国内直接访问会出现**下载速度极慢、连接超时、甚至无法访问**的问题，严重影响数据集加载；
- **提升稳定性**：`hf-mirror.com` 是国内同步的完整镜像，1:1保留官方所有数据集与模型，能大幅提升下载速度，保证国内用户稳定使用；
- **无侵入配置**：通过环境变量全局生效，无需修改后续业务代码，实现一键加速。

---

### 代码2：`dataset = datasets.load_dataset("bentrevett/multi30k")`
#### 代码作用
这是Hugging Face `datasets`库的核心API，完成了完整的数据集加载流程：
1.  **定位数据集**：根据数据集ID `bentrevett/multi30k`，从配置好的镜像源中定位到经典的**英德机器翻译平行语料数据集**；
2.  **下载与缓存**：自动下载数据集原始文件，并缓存到本地，后续运行无需重复下载；
3.  **格式封装**：将原始数据解析为 `DatasetDict` 格式，自动划分好 `train`/`validation`/`test` 三个标准子集；
4.  **赋值存储**：将数据集对象赋值给变量 `dataset`，供后续数据拆分、预处理、模型训练使用。

#### 补充说明
`multi30k` 是seq2seq机器翻译任务的标准基准数据集，包含约3万条英德平行句子对，是NLP序列建模教学的经典入门数据集。

In [3]:
from datasets import load_dataset

# 方法1：用正确的相对路径（从seq2seq文件夹回到上级，再找multi30k）
dataset = load_dataset("json", data_files={
    "train": "../multi30k/train.jsonl",
    "validation": "../multi30k/val.jsonl",
    "test": "../multi30k/test.jsonl"
})

# 方法2：用绝对路径（复制报错里的路径，直接用）
# dataset = load_dataset("json", data_files={
#     "train": "C:/Users/34065/Desktop/seq2seq_project/multi30k/train.jsonl",
#     "validation": "C:/Users/34065/Desktop/seq2seq_project/multi30k/val.jsonl",
#     "test": "C:/Users/34065/Desktop/seq2seq_project/multi30k/test.jsonl"
# })

print("✅ 数据集加载成功！")
print(dataset)

✅ 数据集加载成功！
DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})


## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [5]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [6]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [7]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [8]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


### `tokenize_example` 函数作用
该函数是**数据集预处理的核心分词工具**，用于对英德平行语料进行标准化处理，为后续模型训练做准备，核心功能如下：
1.  **分词处理**：调用spaCy分词器，分别对英语(`en`)和德语(`de`)句子进行分词，提取单词文本
2.  **长度截断**：通过`max_length`参数截断过长句子，统一序列长度，避免模型输入溢出
3.  **大小写归一化**：`lower=True`时将所有单词转为小写，减少词汇表规模，提升模型泛化性
4.  **特殊标记添加**：在句子首尾添加`<sos>`（起始标记）和`<eos>`（结束标记），告知模型句子的起止位置
5.  **格式封装**：返回分词后的`en_tokens`和`de_tokens`，适配Hugging Face Dataset的处理格式

最终实现：将原始文本句子 → 标准化分词序列，为词表构建、模型输入做铺垫。

In [9]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


### 一、`<sos>` 和 `<eos>` 的含义
- `<sos>`（Start Of Sequence，序列起始标记）：添加在句子的**开头**，作用是告诉模型「一个新的句子从这里开始」，是序列生成任务的起始信号。
- `<eos>`（End Of Sequence，序列结束标记）：添加在句子的**末尾**，作用是告诉模型「一个句子到这里结束」，用于标识句子边界，辅助模型生成完整的翻译结果。
- 核心作用：在 Seq2Seq 机器翻译模型中，标记输入/输出序列的起止位置，是编码器、解码器准确处理序列的关键特殊符号。

---

### 二、`map` 函数的作用
`map` 是 Hugging Face `datasets` 库的核心数据处理方法，作用如下：
1.  **批量处理**：对数据集（`train_data`/`valid_data`/`test_data`）的**每一条样本**，批量执行 `tokenize_example` 分词函数，无需手动循环。
2.  **参数统一**：通过 `fn_kwargs` 批量传入分词器、最大长度、大小写转换、特殊标记等参数，保证全数据集处理规则一致。
3.  **格式转换**：将原始文本数据，转换为带分词结果、特殊标记的标准化格式，生成新的 `en_tokens`/`de_tokens` 特征。
4.  **高效执行**：底层支持并行加速，大幅提升大规模数据集（如 multi30k 的 29000 条训练样本）的预处理效率。

最终实现：对训练/验证/测试集完成全量分词、加特殊标记的预处理，为后续构建词表、模型训练做准备。

In [10]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [11]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

In [12]:
# 特殊标记定义（复用之前的变量，避免重复定义）
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
# sos_token、eos_token 之前已经定义过，直接复用
special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

# 构建英语词汇表（用 train_data 的 en_tokens 特征）
en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

# 构建德语词汇表（用 train_data 的 de_tokens 特征）
de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

# 设置默认索引为 <unk>（处理未知词）
en_vocab.set_default_index(en_vocab[unk_token])
de_vocab.set_default_index(de_vocab[unk_token])

print(f"✅ 英语词汇表构建完成，大小：{len(en_vocab)}")
print(f"✅ 德语词汇表构建完成，大小：{len(de_vocab)}")

✅ 英语词汇表构建完成，大小：5893
✅ 德语词汇表构建完成，大小：7853


## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [13]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [14]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [15]:
en_vocab["the"]

7

In [16]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [17]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [18]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [19]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


### 为什么单词 "crime" 会被转换为 `<unk>`
`<unk>`（Unknown，未知词）是词汇表中的特殊标记，用于表示**词表中不存在的单词**，"crime" 被转换为 `<unk>` 的核心原因如下：
1.  **低频词过滤**：构建词表时设置了 `min_freq=2`（最低出现频率为2），"crime" 在训练集中的出现次数**小于2**，被过滤出了词汇表
2.  **OOV 处理机制**：当模型/词表遇到未收录的单词（Out Of Vocabulary, OOV）时，会统一映射为 `<unk>`，避免因未知词导致程序报错
3.  **泛化性保障**：通过 `<unk>` 标记，模型可以处理训练集外的新单词，提升模型的泛化能力，不会因罕见词崩溃

简单来说：**"crime" 太少见，没进词表，所以被标记为「未知词」**。

### 1. `numericalize_example` 函数的作用
该函数是**文本数值化的核心工具**，用于将分词后的 Token 序列，转换为词汇表对应的数字索引，核心功能如下：
1.  **索引映射**：调用 `vocab.lookup_indices()` 方法，将英语/德语的 Token 列表，批量转换为词表对应的整数索引列表
2.  **格式封装**：返回 `en_ids`（英语索引）和 `de_ids`（德语索引），适配 Hugging Face Dataset 的处理格式
3.  **标准化输入**：将字符串形式的文本，转换为神经网络可直接处理的数字张量，为模型训练做准备

### 2. `map` 函数的作用
`map` 是 Hugging Face `datasets` 库的核心数据处理方法，作用如下：
1.  **批量处理**：对训练集/验证集/测试集的**每一条样本**，批量执行 `numericalize_example` 函数，无需手动循环
2.  **参数统一**：通过 `fn_kwargs` 批量传入英/德词表，保证全数据集数值化规则一致
3.  **高效执行**：底层支持并行加速，大幅提升大规模数据集的预处理效率
4.  **格式转换**：将分词后的文本数据集，转换为带数字索引的标准化数据集，生成新的 `en_ids`/`de_ids` 特征

最终实现：对全量数据集完成**字符串→数字**的转换，为后续 DataLoader 构建、模型训练做铺垫。

In [20]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [21]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [22]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [23]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [24]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

### 1. `get_collate_fn` 函数作用
`get_collate_fn` 是**自定义批次整理函数的工厂函数**，核心作用是生成适配 `DataLoader` 的 `collate_fn`，解决「变长序列批量处理」的问题：
1.  **参数接收**：接收 `pad_index`（填充标记的索引），用于统一填充值
2.  **内部 `collate_fn` 逻辑**：
    - 提取批次中所有样本的英语/德语数字索引序列
    - 调用 `nn.utils.rnn.pad_sequence` 对序列进行**填充对齐**，将不同长度的句子补成相同长度，填充值为 `pad_index`
    - 封装为字典格式返回，适配数据集结构
3.  **核心价值**：解决RNN/Seq2Seq模型对「等长张量输入」的要求，是变长文本批量训练的关键工具

---

### 2. `get_data_loader` 函数作用
`get_data_loader` 是**数据加载器的封装函数**，用于快速创建训练/验证/测试集的 `DataLoader`：
1.  **参数配置**：接收数据集、批次大小、填充索引、是否打乱等参数
2.  **核心逻辑**：
    - 调用 `get_collate_fn` 生成自定义的 `collate_fn`
    - 实例化 `torch.utils.data.DataLoader`，传入数据集、批次大小、`collate_fn`、打乱规则
    - 返回配置好的 `DataLoader` 对象
3.  **核心价值**：
    - 统一数据加载逻辑，避免重复代码
    - 支持训练集打乱（`shuffle=True`）、验证/测试集不打乱（`shuffle=False`），符合模型训练规范
    - 生成的 `DataLoader` 可直接用于模型训练，实现批次迭代、数据并行加载

---

### 3. 整体流程总结
两个函数配合，完成了**数据准备的最后一步**：
1.  `get_collate_fn` 解决「变长序列对齐」问题
2.  `get_data_loader` 封装「数据加载逻辑」，生成可直接用于训练的批次数据
3.  最终输出 `train_data_loader`/`valid_data_loader`/`test_data_loader`，为后续Seq2Seq模型训练提供数据支撑

In [25]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [26]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [27]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

### Encoder 类代码解释

Encoder 是 Seq2Seq 模型中的**编码器**，负责把输入的英语句子编码成语义向量，传递给解码器进行翻译。

1. **作用**
   - 接收英语句子的数字索引序列
   - 通过词嵌入层将数字转为向量
   - 通过 LSTM 提取句子的语义信息
   - 输出最后的隐藏状态和细胞状态，作为解码器的初始输入

2. **参数说明**
   - input_dim：英语词汇表大小
   - embedding_dim：词向量维度
   - hidden_dim：LSTM 隐藏层维度
   - n_layers：LSTM 层数
   - dropout：防止过拟合的随机失活率

3. **网络结构**
   - Embedding：词嵌入层，把单词索引转为向量
   - LSTM：循环神经网络，处理序列信息
   - Dropout：随机失活，防止过拟合

4. **forward 流程**
   - 输入英语句子 → 词嵌入 → Dropout → LSTM → 输出语义状态
   - 最终返回 hidden 和 cell，作为解码器的初始状态

In [28]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

### Decoder（解码器）工作流程详解
Decoder 是 Seq2Seq 翻译模型的**解码端**，核心作用是接收编码器输出的语义状态，逐词生成目标语言（德语）的翻译结果，完整流程如下：

---
#### 1. 初始化与核心组件
Decoder 继承自 `nn.Module`，初始化时定义 5 个核心参数与 4 个网络层：
- **参数**：`output_dim`（目标语言词汇表大小）、`embedding_dim`（词嵌入维度）、`hidden_dim`（LSTM 隐藏层维度，需与编码器一致）、`n_layers`（LSTM 层数，需与编码器一致）、`dropout`（Dropout 失活率）
- **网络层**：
  - `nn.Embedding`：目标语言词嵌入层，将单词索引转为稠密语义向量
  - `nn.LSTM`：2 层堆叠 LSTM，用于时序解码，生成目标语言序列
  - `nn.Linear`：全连接输出层，将 LSTM 隐藏状态映射为词汇表概率分布
  - `nn.Dropout`：Dropout 层，抑制过拟合

---
#### 2. forward 单次解码流程（逐词生成）
Decoder 采用**自回归解码**，每次输入一个已生成的单词，输出下一个单词的预测，单步流程：
1.  **输入处理**：接收 3 个输入
    - `input`：上一个时间步生成的目标语言单词索引（初始为 `<sos>` 起始标记）
    - `hidden`/`cell`：编码器输出的最后时间步隐藏/细胞状态（作为解码器初始上下文）
2.  **维度调整**：通过 `unsqueeze(0)` 将输入维度从 `[batch_size]` 转为 `[1, batch_size]`，适配 LSTM 序列输入要求
3.  **词嵌入与 Dropout**：将单词索引转为词向量，施加 Dropout 正则化，输出 `[1, batch_size, embedding_dim]` 的嵌入序列
4.  **LSTM 解码**：将嵌入序列与编码器的 `hidden`/`cell` 输入 LSTM，输出当前时间步的隐藏状态 `output`，以及更新后的 `hidden`/`cell`（用于下一个时间步）
5.  **维度压缩与预测**：通过 `squeeze(0)` 压缩序列维度，将 `output` 输入全连接层 `fc_out`，生成目标语言词汇表的概率分布 `prediction`（形状 `[batch_size, output_dim]`）
6.  **返回结果**：返回当前单词的预测 `prediction`，以及更新后的 `hidden`/`cell`（用于下一个时间步的解码）

---
#### 3. 整体解码逻辑
Decoder 以编码器的语义状态为初始上下文，从 `<sos>` 标记开始，**逐词迭代生成**目标语言序列，直到生成 `<eos>` 结束标记，最终完成完整句子的翻译，实现「源语言语义→目标语言翻译」的转换。

In [29]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

### Seq2Seq 类代码与工作流程解释
Seq2Seq 是序列到序列（Sequence-to-Sequence）模型的封装类，用于将编码器（Encoder）和解码器（Decoder）组合，实现端到端的机器翻译，其核心逻辑围绕 **`__init__` 初始化**、**`forward` 前向传播** 与 **Teacher Forcing 机制** 展开。

---
#### 1. `__init__` 初始化方法
**核心作用**：将编码器、解码器与计算设备绑定，完成基础配置与一致性校验
- **输入参数**：
  - `encoder`：已定义好的编码器实例（负责源语言语义编码）
  - `decoder`：已定义好的解码器实例（负责目标语言序列生成）
  - `device`：计算设备（CPU/GPU），用于张量迁移
- **核心逻辑**：
  1.  保存编码器、解码器与设备信息；
  2.  双 `assert` 校验：强制编码器与解码器的 **隐藏层维度（hidden_dim）**、**LSTM 层数（n_layers）** 必须一致，保证语义状态传递的兼容性，否则直接抛出报错。

---
#### 2. `forward` 前向传播流程（核心训练逻辑）
**核心作用**：完成单轮训练的前向计算，输入源语言（src）与目标语言（trg），输出解码器的预测结果，实现端到端训练。
**输入参数**：
- `src`：源语言（英语）索引序列，形状 `[src_length, batch_size]`
- `trg`：目标语言（德语）索引序列，形状 `[trg_length, batch_size]`
- `teacher_forcing_ratio`：教师强制（Teacher Forcing）的概率（0~1之间）

**单步流程详解**：
1.  **初始化维度与变量**：
    - 提取批次大小（`batch_size`）、目标语言长度（`trg_length`）、目标词汇表大小（`trg_vocab_size`）；
    - 初始化全零张量 `outputs`，用于存储每个时间步的预测结果，形状 `[trg_length, batch_size, trg_vocab_size]`。
2.  **编码器编码**：
    - 将源语言序列 `src` 输入编码器，得到解码器的初始隐藏状态（`hidden`）与细胞状态（`cell`），即 `hidden, cell = self.encoder(src)`。
3.  **解码器初始输入**：
    - 解码器的第一个输入固定为目标语言的 `<sos>` 标记（`trg[0, :]`），作为序列生成的起点。
4.  **时序解码 + Teacher Forcing 循环**：
    - 遍历目标语言序列的每个时间步（从 1 到 `trg_length-1`），执行单步解码：
      1.  **输入当前 token**：根据 `teacher_forcing_ratio` 决定输入来源：
          - 以概率 `teacher_forcing_ratio`：使用**真实标签（ground-truth）**作为输入（`trg[t, :]`），即教师强制，快速稳定训练；
          - 以概率 `1-teacher_forcing_ratio`：使用**解码器上一步的预测结果**作为输入，模拟推理阶段的生成逻辑；
      2.  **解码预测**：将当前输入 + 上一步的 `hidden`/`cell` 输入解码器，得到当前 token 的预测 `output`，以及更新后的 `hidden`/`cell`；
      3.  **存储预测**：将当前时间步的预测存入 `outputs[t, :, :]`。
5.  **返回结果**：返回所有时间步的预测结果 `outputs`，形状 `[trg_length, batch_size, trg_vocab_size]`，后续可用于计算损失（如交叉熵）。

---
#### 3. Teacher Forcing 机制（核心创新点）
**定义**：序列生成任务中的一种训练策略，核心是**用真实标签（ground-truth）的上一步输入，替代解码器上一步的预测输入**。
**作用与价值**：
- 解决训练初期解码器预测误差累积问题：若初始用预测结果作为输入，误差会快速扩散，导致训练不稳定；
- 加速模型收敛：真实标签的语义更准确，能让解码器更快学习到序列映射规律，提升训练效率；
- 平衡训练稳定性与生成真实性：通过 `teacher_forcing_ratio` 动态调整真实标签与预测输入的比例，兼顾训练速度与泛化能力。

In [30]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [31]:
# 超参数配置
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

# 权重初始化
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

权重初始化

In [32]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [33]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [34]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [35]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [36]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 将模型设置为训练模式（启用 dropout、batch norm 等训练专属层）
    model.train()
    # 初始化本轮 epoch 的总损失为 0
    epoch_loss = 0
    # 遍历数据加载器中的每一个批次数据
    for i, batch in enumerate(data_loader):
        # 从批次中取出源语言（德语）索引序列，迁移到指定计算设备（CPU/GPU）
        src = batch["de_ids"].to(device)
        # 从批次中取出目标语言（英语）索引序列，迁移到指定计算设备（CPU/GPU）
        trg = batch["en_ids"].to(device)
        # src 形状: [src_length, batch_size]，trg 形状: [trg_length, batch_size]
        
        # 清空优化器的梯度缓存（防止梯度累积）
        optimizer.zero_grad()
        # 前向传播：将源语言、目标语言输入模型，得到模型预测输出
        output = model(src, trg, teacher_forcing_ratio)
        # output 形状: [trg_length, batch_size, trg_vocab_size]
        
        # 获取目标语言词汇表的维度（即 output 的最后一维大小）
        output_dim = output.shape[-1]
        # 对输出进行维度变换：去掉 <sos> 标记，将序列与批次维度合并，适配损失函数输入
        output = output[1:].view(-1, output_dim)
        # output 形状: [(trg_length - 1) * batch_size, trg_vocab_size]
        
        # 对目标语言标签进行维度变换：去掉 <sos> 标记，与输出维度对齐
        trg = trg[1:].view(-1)
        # trg 形状: [(trg_length - 1) * batch_size]
        
        # 计算模型预测与真实标签之间的损失
        loss = criterion(output, trg)
        # 反向传播：计算梯度
        loss.backward()
        # 梯度裁剪：防止梯度爆炸，将梯度范数限制在 clip 范围内
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        # 优化器更新模型参数
        optimizer.step()
        # 累加当前批次的损失到 epoch 总损失中
        epoch_loss += loss.item()
    # 返回本轮 epoch 的平均损失（总损失除以批次数量）
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [37]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)
            trg = batch["en_ids"].to(device)
            # src = [src length, batch size]
            # trg = [trg length, batch size]
            output = model(src, trg, 0)  # turn off teacher forcing
            # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)
            # trg = [(trg length - 1) * batch size]
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

# 模型训练

In [38]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [10:29<00:00, 629.36s/it]

	Train Loss:   5.026 | Train PPL: 152.277
	Valid Loss:   4.861 | Valid PPL: 129.150


# 模型验证

In [39]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [40]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [41]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [42]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [43]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'a',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '<eos>']